# Debugging

The initial runs had some problems, including claiming that the most remote point in the WMNF was like 0.08 miles from a road. This is wrong, Owl's Head is 3 miles from the nearest road, so clearly, there is a bug. My suspicion is that the road filtering wasn't aggressive enough. 

In [9]:
import json
import pprint

In [13]:
with open("wmnf_roads.json", 'r') as infile:
    wmnf_roads = json.load(infile)

In [14]:
# What are all the possible values of a "highway" tag? 
tags = []
for elem in wmnf_roads["elements"]:
    if elem["type"] == "way":
        if "highway" in elem["tags"].keys():
            tags.append(elem["tags"]["highway"])


In [15]:
counts = {}
for t in tags:
    if t in counts.keys():
        counts[t] += 1
    else:
        counts[t] = 1
pprint.pprint(counts, sort_dicts=True)


{'bridleway': 21,
 'busway': 2,
 'construction': 3,
 'living_street': 2,
 'motorway': 374,
 'motorway_link': 272,
 'pedestrian': 13,
 'primary': 616,
 'primary_link': 12,
 'raceway': 23,
 'residential': 8694,
 'rest_area': 7,
 'secondary': 697,
 'secondary_link': 12,
 'steps': 85,
 'tertiary': 519,
 'tertiary_link': 7,
 'trunk': 339,
 'trunk_link': 20,
 'unclassified': 1412}


There are a lot of things that are marked as "unclassified", and a lot of things marked as "track". The valid tags are here (https://wiki.openstreetmap.org/wiki/Key:highway). One thing of note is that "highway: service" has a "service" tag to tell what kind of service. 

The highway type "track" is for agricultural or forestry use, so dropping those would probably be good for taking out forest service roads. 

In [ ]:
# What services are used?
svcs = []
for elem in wmnf_roads["elements"]:
    if elem["type"] == "way":
        if "service" in elem["tags"].keys():
            svcs.append(elem["tags"]["service"])

counts = {}
for t in svcs:
    if t in counts.keys():
        counts[t] += 1
    else:
        counts[t] = 1
pprint.pprint(counts, sort_dicts=True)

{'alley': 14,
 'bridge': 1,
 'drive-through': 106,
 'driveway': 14638,
 'emergency_access': 51,
 'fuel': 2,
 'parking': 2,
 'parking_aisle': 1308,
 'private_railroad_right_of_way': 1,
 'resource extraction': 1,
 'resource_extraction': 90,
 'rest_area': 1,
 'slipway': 20,
 'spur': 1,
 'utility': 3}


The vast majority of the service roads are driveways and parking aisles. These can safely be eliminiated from consideration, which makes the computation of the Voronoi diagram easier. 